In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [15]:
RAW_PATH = Path("../data/raw/energydata_complete.csv")
TARGET_COLUMN = "Appliances"

print(RAW_PATH.exists())

True


In [18]:
df = pd.read_csv(RAW_PATH)
df.head().T

,0,1,2,3,4
date,11-01-2016 17:00,11-01-2016 17:10,11-01-2016 17:20,11-01-2016 17:30,11-01-2016 17:40
Appliances,60,60,50,50,60
lights,30,30,30,40,40
T1,19.89,19.89,19.89,19.89,19.89
RH_1,47.596667,46.693333,46.3,46.066667,46.333333
T2,19.2,19.2,19.2,19.2,19.2
RH_2,44.79,44.7225,44.626667,44.59,44.53
T3,19.79,19.79,19.79,19.79,19.79
RH_3,44.73,44.79,44.933333,45.0,45.0
T4,19.0,19.0,18.926667,18.89,18.89


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 45 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   date                        19735 non-null  datetime64[ns]
 1   Appliances                  19735 non-null  int64         
 2   lights                      19735 non-null  int64         
 3   T1                          19735 non-null  float64       
 4   RH_1                        19735 non-null  float64       
 5   T2                          19735 non-null  float64       
 6   RH_2                        19735 non-null  float64       
 7   T3                          19735 non-null  float64       
 8   RH_3                        19735 non-null  float64       
 9   T4                          19735 non-null  float64       
 10  RH_4                        19735 non-null  float64       
 11  T5                          19735 non-null  float64   

In [19]:
df.columns.tolist()

['date',
 'Appliances',
 'lights',
 'T1',
 'RH_1',
 'T2',
 'RH_2',
 'T3',
 'RH_3',
 'T4',
 'RH_4',
 'T5',
 'RH_5',
 'T6',
 'RH_6',
 'T7',
 'RH_7',
 'T8',
 'RH_8',
 'T9',
 'RH_9',
 'T_out',
 'Press_mm_hg',
 'RH_out',
 'Windspeed',
 'Visibility',
 'Tdewpoint',
 'rv1',
 'rv2']

In [21]:
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y %H:%M")
df = df.sort_values("date").reset_index(drop=True)

df[["date", "Appliances"]].head()

,date,Appliances
0,2016-01-11 17:00:00,60
1,2016-01-11 17:10:00,60
2,2016-01-11 17:20:00,50
3,2016-01-11 17:30:00,50
4,2016-01-11 17:40:00,60


In [22]:
print(df["date"].min())
print(df["date"].max())
print(df["date"].is_monotonic_increasing)

2016-01-11 17:00:00
2016-05-27 18:00:00
True


In [28]:
df["hour"] = df["date"].dt.hour
df["dayofweek"] = df["date"].dt.dayofweek
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

df[["date", "hour", "dayofweek", "is_weekend"]].head(10)

print(df['hour'].value_counts())
print(df['dayofweek'].value_counts())
print(df['is_weekend'].value_counts())

hour
17    828
18    823
19    822
20    822
21    822
22    822
23    822
0     822
1     822
2     822
3     822
4     822
5     822
6     822
7     822
8     822
9     822
10    822
11    822
12    822
13    822
14    822
15    822
16    822
Name: count, dtype: int64
dayofweek
1    2880
2    2880
3    2880
4    2845
0    2778
5    2736
6    2736
Name: count, dtype: int64
is_weekend
0    14263
1     5472
Name: count, dtype: int64


In [29]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

df[["hour", "hour_sin", "hour_cos", "dayofweek", "dayofweek_sin", "dayofweek_cos"]].head(10)

,hour,hour_sin,hour_cos,dayofweek,dayofweek_sin,dayofweek_cos
0,17,-0.965926,-2.588190e-01,0,0.0,1.0
1,17,-0.965926,-2.588190e-01,0,0.0,1.0
2,17,-0.965926,-2.588190e-01,0,0.0,1.0
3,17,-0.965926,-2.588190e-01,0,0.0,1.0
4,17,-0.965926,-2.588190e-01,0,0.0,1.0
5,17,-0.965926,-2.588190e-01,0,0.0,1.0
6,18,-1.000000,-1.836970e-16,0,0.0,1.0
7,18,-1.000000,-1.836970e-16,0,0.0,1.0
8,18,-1.000000,-1.836970e-16,0,0.0,1.0
9,18,-1.000000,-1.836970e-16,0,0.0,1.0


In [30]:
lag_steps = [1, 3, 6, 12, 24]

for lag in lag_steps:
    df[f"appliances_lag_{lag}"] = df[TARGET_COLUMN].shift(lag)

df[
    [
        "Appliances",
        "appliances_lag_1",
        "appliances_lag_3",
        "appliances_lag_6",
        "appliances_lag_12",
        "appliances_lag_24",
    ]
].head(30)

,Appliances,appliances_lag_1,appliances_lag_3,appliances_lag_6,appliances_lag_12,appliances_lag_24
0,60,NaN,NaN,NaN,NaN,NaN
1,60,60.0,NaN,NaN,NaN,NaN
2,50,60.0,NaN,NaN,NaN,NaN
3,50,50.0,60.0,NaN,NaN,NaN
4,60,50.0,60.0,NaN,NaN,NaN
5,50,60.0,50.0,NaN,NaN,NaN
6,60,50.0,50.0,60.0,NaN,NaN
7,60,60.0,60.0,60.0,NaN,NaN
8,60,60.0,50.0,50.0,NaN,NaN
9,70,60.0,60.0,50.0,NaN,NaN


In [31]:
shifted_target = df[TARGET_COLUMN].shift(1)

df["appliances_rolling_mean_6"] = shifted_target.rolling(window=6).mean()
df["appliances_rolling_std_6"] = shifted_target.rolling(window=6).std()

df["appliances_rolling_mean_24"] = shifted_target.rolling(window=24).mean()
df["appliances_rolling_std_24"] = shifted_target.rolling(window=24).std()

df[
    [
        "Appliances",
        "appliances_rolling_mean_6",
        "appliances_rolling_std_6",
        "appliances_rolling_mean_24",
        "appliances_rolling_std_24",
    ]
].head(35)

,Appliances,appliances_rolling_mean_6,appliances_rolling_std_6,appliances_rolling_mean_24,appliances_rolling_std_24
0,60,NaN,NaN,NaN,NaN
1,60,NaN,NaN,NaN,NaN
2,50,NaN,NaN,NaN,NaN
3,50,NaN,NaN,NaN,NaN
4,60,NaN,NaN,NaN,NaN
5,50,NaN,NaN,NaN,NaN
6,60,55.000000,5.477226,NaN,NaN
7,60,55.000000,5.477226,NaN,NaN
8,60,55.000000,5.477226,NaN,NaN
9,70,56.666667,5.163978,NaN,NaN


In [32]:
sensor_columns = [
    "T1", "RH_1",
    "T2", "RH_2",
    "T3", "RH_3",
    "T4", "RH_4",
    "T5", "RH_5",
    "T6", "RH_6",
    "T7", "RH_7",
    "T8", "RH_8",
    "T9", "RH_9",
]

weather_columns = [
    "T_out",
    "RH_out",
    "Windspeed",
    "Visibility",
    "Tdewpoint",
    "Press_mm_hg",
]

time_columns = [
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",
]

lag_columns = [
    "appliances_lag_1",
    "appliances_lag_3",
    "appliances_lag_6",
    "appliances_lag_12",
    "appliances_lag_24",
]

rolling_columns = [
    "appliances_rolling_mean_6",
    "appliances_rolling_std_6",
    "appliances_rolling_mean_24",
    "appliances_rolling_std_24",
]

tabular_feature_columns = (
    sensor_columns
    + weather_columns
    + time_columns
    + lag_columns
    + rolling_columns
)

lstm_feature_columns = [TARGET_COLUMN] + tabular_feature_columns

print("tabular feature count:", len(tabular_feature_columns))
print("lstm feature count:", len(lstm_feature_columns))


tabular feature count: 38
lstm feature count: 39


In [33]:
working_df = df[["date", TARGET_COLUMN] + tabular_feature_columns].dropna().reset_index(drop=True)
working_df.head()

,date,Appliances,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,dayofweek_cos,appliances_lag_1,appliances_lag_3,appliances_lag_6,appliances_lag_12,appliances_lag_24,appliances_rolling_mean_6,appliances_rolling_std_6,appliances_rolling_mean_24,appliances_rolling_std_24
0,2016-01-11 21:00:00,110,21.133333,46.060000,20.426667,44.760000,20.29,46.433333,19.390000,48.193333,...,1.0,110.0,190.0,80.0,430.0,60.0,125.000000,37.282704,132.500000,129.051725
1,2016-01-11 21:10:00,110,21.200000,45.800000,20.500000,44.760000,20.39,46.223333,19.390000,47.800000,...,1.0,110.0,110.0,140.0,250.0,60.0,130.000000,31.622777,134.583333,128.231420
2,2016-01-11 21:20:00,100,21.290000,45.900000,20.533333,45.090000,20.39,46.090000,19.390000,47.560000,...,1.0,110.0,110.0,120.0,100.0,50.0,125.000000,32.093613,136.666667,127.370281
3,2016-01-11 21:30:00,100,21.356667,45.826667,20.666667,45.163333,20.39,46.090000,19.390000,47.500000,...,1.0,100.0,110.0,190.0,100.0,50.0,121.666667,33.714487,138.750000,126.295461
4,2016-01-11 21:40:00,100,21.390000,45.690000,20.700000,45.060000,20.39,46.090000,19.426667,47.993333,...,1.0,100.0,110.0,110.0,90.0,60.0,106.666667,5.163978,140.833333,125.175239


In [34]:
train_size = int(len(working_df) * 0.8)
train_df = working_df.iloc[:train_size].copy()
test_df = working_df.iloc[train_size:].copy()

print(len(train_df), len(test_df))
print(train_df["date"].min(), train_df["date"].max())
print(test_df["date"].min(), test_df["date"].max())

15768 3943
2016-01-11 21:00:00 2016-04-30 08:50:00
2016-04-30 09:00:00 2016-05-27 18:00:00


In [35]:
y_true = test_df["Appliances"].values
y_pred = test_df["appliances_lag_1"].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("Naive MAE:", round(mae, 2))
print("Naive RMSE:", round(rmse, 2))

Naive MAE: 26.53
Naive RMSE: 66.23


In [37]:
X_train_tabular = train_df[tabular_feature_columns]
y_train = train_df["Appliances"]

X_test_tabular = test_df[tabular_feature_columns]
y_test = test_df["Appliances"]

print(X_train_tabular.shape, y_train.shape)
print(X_test_tabular.shape, y_test.shape)
X_train_tabular.head()

(15768, 38) (15768,)
(3943, 38) (3943,)


,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,...,dayofweek_cos,appliances_lag_1,appliances_lag_3,appliances_lag_6,appliances_lag_12,appliances_lag_24,appliances_rolling_mean_6,appliances_rolling_std_6,appliances_rolling_mean_24,appliances_rolling_std_24
0,21.133333,46.060000,20.426667,44.760000,20.29,46.433333,19.390000,48.193333,17.760000,82.46,...,1.0,110.0,190.0,80.0,430.0,60.0,125.000000,37.282704,132.500000,129.051725
1,21.200000,45.800000,20.500000,44.760000,20.39,46.223333,19.390000,47.800000,18.356667,82.59,...,1.0,110.0,110.0,140.0,250.0,60.0,130.000000,31.622777,134.583333,128.231420
2,21.290000,45.900000,20.533333,45.090000,20.39,46.090000,19.390000,47.560000,18.356667,70.99,...,1.0,110.0,110.0,120.0,100.0,50.0,125.000000,32.093613,136.666667,127.370281
3,21.356667,45.826667,20.666667,45.163333,20.39,46.090000,19.390000,47.500000,18.600000,62.43,...,1.0,100.0,110.0,190.0,100.0,50.0,121.666667,33.714487,138.750000,126.295461
4,21.390000,45.690000,20.700000,45.060000,20.39,46.090000,19.426667,47.993333,18.666667,59.03,...,1.0,100.0,110.0,110.0,90.0,60.0,106.666667,5.163978,140.833333,125.175239


In [38]:
scaler = StandardScaler()

train_lstm_scaled = scaler.fit_transform(train_df[lstm_feature_columns])
test_lstm_scaled = scaler.transform(test_df[lstm_feature_columns])

print(train_lstm_scaled.shape)
print(test_lstm_scaled.shape)

(15768, 39)
(3943, 39)


In [39]:
SEQ_LEN = 24

def make_windows(data, seq_len=24, target_col_idx=0):
    xs = []
    ys = []
    for idx in range(len(data) - seq_len):
        xs.append(data[idx:idx + seq_len])
        ys.append(data[idx + seq_len, target_col_idx])
    return np.array(xs), np.array(ys)

x_train, y_train_seq = make_windows(train_lstm_scaled, seq_len=SEQ_LEN)
x_test, y_test_seq = make_windows(test_lstm_scaled, seq_len=SEQ_LEN)

print("x_train:", x_train.shape)
print("y_train_seq:", y_train_seq.shape)
print("x_test:", x_test.shape)
print("y_test_seq:", y_test_seq.shape)
print("sample window shape:", x_train[0].shape)

x_train: (15744, 24, 39)
y_train_seq: (15744,)
x_test: (3919, 24, 39)
y_test_seq: (3919,)
sample window shape: (24, 39)
